# 09 — Final test evaluation (result freeze)
- **Run only after every model to be reported is trained and frozen in DECISION_LOG** (E1 D021, E2 D023, E3, and E4 if it is run). This is the only notebook that reads the test split (CONSTITUTION C3, D018).
- **Each checkpoint is evaluated once** (D026): `{id}_best.pt` is scored on the 707 test images; a re-run reuses the stored result, and the notebook stops if a checkpoint changed after its evaluation. Nothing may be tuned on these numbers.
- **Per experiment** (`04_Results/experiments/{id}/`): `{id}_metrics.json`, `{id}_predictions.csv`, `{id}_per_class_metrics.csv`, `{id}_confusion_matrix.png`, `{id}_errors.csv` (misclassified images with empty columns for the error analysis).
- **Comparison** (`04_Results/final/`): `model_comparison.csv`, `per_class_f1_comparison.csv`, error examples of the primary model.
- **Primary model for error analysis:** the one with the best **validation** macro-F1, never chosen by test results.
- **Runtime:** GPU optional (inference only).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KNV-DSEB/realwaste-classification.git"
REPO_DIR = Path("/content/realwaste-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Forget src.* modules imported earlier in this session, so a re-run uses the code just pulled.
for module_name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[module_name]

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image

from src.dataset import load_frozen_split
from src.final_eval import comparison_table, evaluate_on_test, experiment_dirs, per_class_f1_table
from src.utils import environment_info, get_paths, load_config

cfg = load_config(REPO_DIR / "configs" / "config.yaml")
paths = get_paths(cfg)
FINAL_DIR = paths.result_dir / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp = device.type == "cuda"
print(json.dumps(environment_info(), indent=2))
print("device:", device)

split_df = load_frozen_split(cfg)  # SHA256-verified frozen grouped split (D016)
print({s: int(n) for s, n in split_df["split"].value_counts().items()})

In [ ]:
REQUIRED = ["E1", "E2", "E3"]
OPTIONAL = ["E4"]

results = []
for experiment_id in REQUIRED + OPTIONAL:
    _, result_dir = experiment_dirs(cfg, experiment_id)
    if not (result_dir / f"{experiment_id}_training_summary.json").exists():
        if experiment_id in REQUIRED:
            raise FileNotFoundError(f"{experiment_id} has no finished training run; train it before the final evaluation.")
        print(f"{experiment_id}: no finished training run, skipped (optional).")
        continue
    results.append(evaluate_on_test(cfg, experiment_id, split_df, device, amp))

In [ ]:
table = comparison_table(results)
table.to_csv(FINAL_DIR / "model_comparison.csv", index=False)
print("Test set (707 images); 'Val Macro F1' is the validation score used for model selection.\n")
print(table.to_string(index=False))

per_class_f1 = per_class_f1_table(cfg, [r["experiment_id"] for r in results])
per_class_f1.to_csv(FINAL_DIR / "per_class_f1_comparison.csv")
print("\nTest F1 per class:\n")
print(per_class_f1.to_string())

In [ ]:
for r in results:
    _, result_dir = experiment_dirs(cfg, r["experiment_id"])
    plt.figure(figsize=(8, 7))
    plt.imshow(plt.imread(result_dir / f"{r['experiment_id']}_confusion_matrix.png"))
    plt.axis("off")
    plt.show()

In [ ]:
# Error analysis starts from the model with the best VALIDATION macro-F1 (never chosen by test results).
primary = max(results, key=lambda r: r["best_val_macro_f1"])
_, result_dir = experiment_dirs(cfg, primary["experiment_id"])
errors = pd.read_csv(result_dir / f"{primary['experiment_id']}_errors.csv")
pair_counts = errors.groupby(["true_class", "predicted_class"]).size().sort_values(ascending=False)
print(f"Primary model: {primary['experiment_id']} {primary['model_name']} - {len(errors)} misclassified test images")
print(pair_counts.head(10).to_string())

image_path = dict(zip(split_df["filepath"], split_df["image_path"]))
sample = pd.concat([errors[(errors["true_class"] == t) & (errors["predicted_class"] == p)].head(4)
                    for t, p in pair_counts.head(6).index])
cols = 6
rows = -(-len(sample) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.9), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, row in zip(axes.flat, sample.itertuples()):
    with Image.open(image_path[row.filepath]) as img:
        ax.imshow(img.convert("RGB"))
    ax.set_title(f"true {row.true_class}\npred {row.predicted_class} ({row.confidence:.2f})", fontsize=7)
plt.tight_layout()
fig.savefig(FINAL_DIR / f"{primary['experiment_id']}_test_error_examples.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Lines for DECISION_LOG (result freeze) and the report.
for r in results:
    print(f"{r['experiment_id']} {r['model_name']}: test accuracy {r['accuracy']:.4f}, macro precision {r['macro_precision']:.4f}, "
          f"macro recall {r['macro_recall']:.4f}, macro-F1 {r['macro_f1']:.4f} (val macro-F1 {r['best_val_macro_f1']:.4f}; "
          f"evaluated {r['evaluated_at']}; checkpoint sha256 {r['checkpoint_sha256'][:12]})")

## Next
1. Save this notebook with outputs to GitHub and send the printed tables and lines above.
2. Record the result freeze in `docs/DECISION_LOG.md`. From now on no model, split or setting changes because of these numbers.
3. Error analysis: in `{id}_errors.csv` of the primary model, fill `failure_mode`, `visual_note` and `reviewer` for 50–100 images, grouped by confusion pair (`templates/error_analysis_template.csv` columns).